In [1]:
import pandas as pd
from pathlib import Path

file_path = Path("../data/processed/model_eligible.csv")
model_eligible = pd.read_csv(file_path)

print(model_eligible.shape)
print(model_eligible["CHURNED"].value_counts())
print(model_eligible.isna().sum())

(2225, 30)
CHURNED
0    1563
1     662
Name: count, dtype: int64
CUSTOMER_ID                    0
SIGNUP_DATE                    0
AGE                            0
GENDER                         0
CITY                           0
MEMBERSHIP_TIER                0
MARKETING_OPT_IN               0
PREFERRED_STORE_ID             0
PREFERRED_CATEGORY             0
HOME_STORE_PRICE_TIER          0
TRANSACTIONS_PRE               0
TOTAL_SPEND_PRE                0
APP_SESSIONS_PRE               0
EMAILS_OPENED_PRE              0
COUPONS_REDEEMED_PRE           0
SUPPORT_TICKETS_PRE            0
TRANSACTIONS_AVG_PRE           0
TOTAL_SPEND_AVG_PRE            0
AVG_BASKET_VALUE_AVG_PRE       0
PROMO_TXN_PCT_AVG_PRE          0
APP_SESSIONS_AVG_PRE           0
EMAILS_OPENED_AVG_PRE          0
SUPPORT_TICKETS_AVG_PRE        0
ACTIVE_MONTHS_PRE              0
RECENT_TRANSACTIONS          449
RECENT_SPEND                 449
RECENT_APP_SESSIONS          449
RECENT_SUPPORT_TICKETS       449
CHURNED    

In [4]:
missing = model_eligible.isna().sum()
print(missing[missing > 0])

RECENT_TRANSACTIONS       449
RECENT_SPEND              449
RECENT_APP_SESSIONS       449
RECENT_SUPPORT_TICKETS    449
dtype: int64


In [5]:
print(model_eligible.dtypes.to_string())

CUSTOMER_ID                    int64
SIGNUP_DATE                      str
AGE                          float64
GENDER                           str
CITY                             str
MEMBERSHIP_TIER                  str
MARKETING_OPT_IN               int64
PREFERRED_STORE_ID             int64
PREFERRED_CATEGORY               str
HOME_STORE_PRICE_TIER            str
TRANSACTIONS_PRE             float64
TOTAL_SPEND_PRE              float64
APP_SESSIONS_PRE             float64
EMAILS_OPENED_PRE            float64
COUPONS_REDEEMED_PRE         float64
SUPPORT_TICKETS_PRE          float64
TRANSACTIONS_AVG_PRE         float64
TOTAL_SPEND_AVG_PRE          float64
AVG_BASKET_VALUE_AVG_PRE     float64
PROMO_TXN_PCT_AVG_PRE        float64
APP_SESSIONS_AVG_PRE         float64
EMAILS_OPENED_AVG_PRE        float64
SUPPORT_TICKETS_AVG_PRE      float64
ACTIVE_MONTHS_PRE            float64
RECENT_TRANSACTIONS          float64
RECENT_SPEND                 float64
RECENT_APP_SESSIONS          float64
R

In [6]:
# Make a copy so the original dataset stays unchanged
df = model_eligible.copy()

# Convert signup date into a datetime
df["SIGNUP_DATE"] = pd.to_datetime(df["SIGNUP_DATE"])

# Create customer tenure in days
df["TENURE_DAYS"] = (
    pd.Timestamp("2024-04-01") - df["SIGNUP_DATE"]
).dt.days

# Remove columns we should not use as model inputs
X = df.drop(columns=[
    "CUSTOMER_ID",
    "SIGNUP_DATE",
    "CHURNED",
    "INSUFFICIENT_HISTORY_FLAG"
])

# Target: what we want to predict
y = df["CHURNED"]

print("Features:", X.shape)
print("Target:", y.shape)
print("Target values:")
print(y.value_counts())

Features: (2225, 27)
Target: (2225,)
Target values:
CHURNED
0    1563
1     662
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining churn rate:")
print(y_train.mean())

print("\nTesting churn rate:")
print(y_test.mean())

Training data: (1780, 27)
Testing data: (445, 27)

Training churn rate:
0.29775280898876405

Testing churn rate:
0.2966292134831461


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

Numeric columns: 22
Categorical columns: 5


In [9]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline

# Baseline model
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

# Logistic Regression model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Compare results
for name, pred in [
    ("Baseline", baseline_pred),
    ("Logistic Regression", y_pred)
]:
    print(f"\n{name}")
    print("Accuracy :", round(accuracy_score(y_test, pred), 3))
    print("Precision:", round(precision_score(y_test, pred, zero_division=0), 3))
    print("Recall   :", round(recall_score(y_test, pred, zero_division=0), 3))
    print("F1-score :", round(f1_score(y_test, pred, zero_division=0), 3))


Baseline
Accuracy : 0.703
Precision: 0.0
Recall   : 0.0
F1-score : 0.0

Logistic Regression
Accuracy : 0.836
Precision: 0.761
Recall   : 0.652
F1-score : 0.702


In [10]:
from sklearn.metrics import confusion_matrix, classification_report

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[286  27]
 [ 46  86]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.91      0.89       313
           1       0.76      0.65      0.70       132

    accuracy                           0.84       445
   macro avg       0.81      0.78      0.79       445
weighted avg       0.83      0.84      0.83       445



In [11]:
# Get predicted churn probabilities
y_prob = model.predict_proba(X_test)[:, 1]

# Predict churn if probability is at least 0.4
y_pred_04 = (y_prob >= 0.4).astype(int)

print(confusion_matrix(y_test, y_pred_04))
print(classification_report(y_test, y_pred_04))

[[272  41]
 [ 37  95]]
              precision    recall  f1-score   support

           0       0.88      0.87      0.87       313
           1       0.70      0.72      0.71       132

    accuracy                           0.82       445
   macro avg       0.79      0.79      0.79       445
weighted avg       0.83      0.82      0.83       445



In [12]:
results = []

for threshold, pred in [(0.5, y_pred), (0.4, y_pred_04)]:
    results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1-score": f1_score(y_test, pred)
    })

results_df = pd.DataFrame(results)
print(results_df.round(3))

   Threshold  Accuracy  Precision  Recall  F1-score
0        0.5     0.836      0.761   0.652     0.702
1        0.4     0.825      0.699   0.720     0.709


In [13]:
recent_cols = [
    "RECENT_TRANSACTIONS",
    "RECENT_SPEND",
    "RECENT_APP_SESSIONS",
    "RECENT_SUPPORT_TICKETS"
]

missing_recent = X[recent_cols].isna().all(axis=1)

print(pd.crosstab(
    missing_recent,
    y,
    rownames=["All recent features missing"],
    colnames=["CHURNED"]
))

CHURNED                         0    1
All recent features missing           
False                        1556  220
True                            7  442


In [14]:
# Remove the four recent-activity features
recent_cols = [
    "RECENT_TRANSACTIONS",
    "RECENT_SPEND",
    "RECENT_APP_SESSIONS",
    "RECENT_SUPPORT_TICKETS"
]

X_without_recent = X.drop(columns=recent_cols)

# Split using the same settings
X_train_nr, X_test_nr, y_train_nr, y_test_nr = train_test_split(
    X_without_recent, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Build a fresh preprocessing pipeline for these columns
num_cols_nr = X_train_nr.select_dtypes(include="number").columns
cat_cols_nr = X_train_nr.select_dtypes(exclude="number").columns

preprocessor_nr = ColumnTransformer([
    ("numeric", numeric_pipeline, num_cols_nr),
    ("categorical", categorical_pipeline, cat_cols_nr)
])

model_nr = Pipeline([
    ("preprocessor", preprocessor_nr),
    ("classifier", LogisticRegression(max_iter=1000))
])

model_nr.fit(X_train_nr, y_train_nr)
pred_nr = model_nr.predict(X_test_nr)

print(classification_report(y_test_nr, pred_nr))

              precision    recall  f1-score   support

           0       0.86      0.93      0.89       313
           1       0.79      0.64      0.70       132

    accuracy                           0.84       445
   macro avg       0.82      0.78      0.80       445
weighted avg       0.84      0.84      0.84       445



In [15]:
print("With recent features:")
print(confusion_matrix(y_test, y_pred))

print("\nWithout recent features:")
print(confusion_matrix(y_test_nr, pred_nr))

With recent features:
[[286  27]
 [ 46  86]]

Without recent features:
[[290  23]
 [ 48  84]]


In [16]:
results_df.to_csv(
    Path("../data/processed/day3_model_comparison.csv"),
    index=False
)

## Final holdout evaluation and provisional threshold

The source labels all describe the same April–June 2024 outcome window, so they cannot support a genuine future-period test. The following stratified 60/20/20 train/validation/test evaluation is an independent random holdout, not temporal validation. The validation set selects the threshold; the test set is used only for final evaluation. A missing-recent-activity indicator is included because the EDA verified that these nulls represent absent January–March activity records. The F1-maximizing threshold is a provisional statistical choice, not a business-approved operating point.

In [17]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

final_df = model_eligible.copy()
recent_cols = [
    "RECENT_TRANSACTIONS",
    "RECENT_SPEND",
    "RECENT_APP_SESSIONS",
    "RECENT_SUPPORT_TICKETS",
]
final_df["RECENT_ACTIVITY_MISSING"] = (
    final_df[recent_cols].isna().all(axis=1).astype(int)
)
X_final = final_df.drop(columns=[
    "CUSTOMER_ID",
    "SIGNUP_DATE",
    "CHURNED",
    "INSUFFICIENT_HISTORY_FLAG",
])
y_final = final_df["CHURNED"]

X_train_validation, X_test_final, y_train_validation, y_test_final = train_test_split(
    X_final, y_final, test_size=0.20, random_state=42, stratify=y_final
)
X_train_final, X_validation_final, y_train_final, y_validation_final = train_test_split(
    X_train_validation, y_train_validation,
    test_size=0.25, random_state=42, stratify=y_train_validation
)

numeric_final = X_train_final.select_dtypes(include="number").columns
categorical_final = X_train_final.select_dtypes(exclude="number").columns
preprocessor_final = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_final),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_final),
])

final_model = Pipeline([
    ("preprocessor", preprocessor_final),
    ("classifier", LogisticRegression(max_iter=2000)),
])
final_model.fit(X_train_final, y_train_final)
validation_probabilities = final_model.predict_proba(X_validation_final)[:, 1]
validation_precision, validation_recall, validation_thresholds = precision_recall_curve(
    y_validation_final, validation_probabilities
)
validation_f1 = (
    2 * validation_precision[:-1] * validation_recall[:-1]
    / (validation_precision[:-1] + validation_recall[:-1] + 1e-12)
)
best_f1 = validation_f1.max()
best_threshold_index = np.flatnonzero(np.isclose(validation_f1, best_f1))[-1]
selected_threshold = float(validation_thresholds[best_threshold_index])

print(f"Train: {len(X_train_final)} | Validation: {len(X_validation_final)} | Test: {len(X_test_final)}")
print(f"Validation-selected threshold (maximum F1): {selected_threshold:.4f}")
print(f"Validation F1 at selected threshold: {best_f1:.3f}")
print("This threshold is provisional; select the production operating point using retention capacity and error costs.")

Train: 1335 | Validation: 445 | Test: 445
Validation-selected threshold (maximum F1): 0.3996
Validation F1 at selected threshold: 0.935
This threshold is provisional; select the production operating point using retention capacity and error costs.


In [18]:
from pathlib import Path
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)

test_probabilities = final_model.predict_proba(X_test_final)[:, 1]
test_prediction_selected = (test_probabilities >= selected_threshold).astype(int)
test_prediction_default = (test_probabilities >= 0.5).astype(int)

final_metrics = []
dummy_final = DummyClassifier(strategy="most_frequent")
dummy_final.fit(X_train_final, y_train_final)
dummy_prediction = dummy_final.predict(X_test_final)

for model_name, prediction, probability in [
    ("Majority baseline", dummy_prediction, None),
    ("Logistic Regression (0.50)", test_prediction_default, test_probabilities),
    (f"Logistic Regression ({selected_threshold:.3f})", test_prediction_selected, test_probabilities),
]:
    row = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test_final, prediction),
        "Precision": precision_score(y_test_final, prediction, zero_division=0),
        "Recall": recall_score(y_test_final, prediction, zero_division=0),
        "F1": f1_score(y_test_final, prediction, zero_division=0),
        "TN": int(confusion_matrix(y_test_final, prediction).ravel()[0]),
        "FP": int(confusion_matrix(y_test_final, prediction).ravel()[1]),
        "FN": int(confusion_matrix(y_test_final, prediction).ravel()[2]),
        "TP": int(confusion_matrix(y_test_final, prediction).ravel()[3]),
    }
    if probability is not None:
        row["PR_AUC"] = average_precision_score(y_test_final, probability)
        row["ROC_AUC"] = roc_auc_score(y_test_final, probability)
        row["Brier_score"] = brier_score_loss(y_test_final, probability)
    else:
        row["PR_AUC"] = np.nan
        row["ROC_AUC"] = np.nan
        row["Brier_score"] = np.nan
    final_metrics.append(row)

final_metrics_df = pd.DataFrame(final_metrics)
display(final_metrics_df.round(3))
print("\nSelected-threshold test confusion matrix [TN, FP; FN, TP]:")
print(confusion_matrix(y_test_final, test_prediction_selected))

capacity_rows = []
ranked_test = pd.DataFrame({
    "actual": y_test_final.to_numpy(),
    "score": test_probabilities,
}).sort_values("score", ascending=False)
total_test_churners = int(ranked_test["actual"].sum())
for capacity in (0.10, 0.20, 0.30):
    target_count = int(np.ceil(len(ranked_test) * capacity))
    captured = int(ranked_test.head(target_count)["actual"].sum())
    capacity_rows.append({
        "Targeted_share": capacity,
        "Customers_targeted": target_count,
        "Precision_at_capacity": captured / target_count,
        "Churners_captured": captured,
        "Churn_recall_at_capacity": captured / total_test_churners,
    })
capacity_df = pd.DataFrame(capacity_rows)
print("\nTest-set outcomes when targeting the highest-risk customers:")
display(capacity_df.round(3))

output_dir = Path("../data/processed")
final_metrics_df.to_csv(output_dir / "day4_holdout_metrics.csv", index=False)
capacity_df.to_csv(output_dir / "day4_capacity_analysis.csv", index=False)
pd.DataFrame({
    "Selected_threshold": [selected_threshold],
    "Validation_F1": [best_f1],
    "Validation_customers": [len(X_validation_final)],
    "Test_customers": [len(X_test_final)],
    "Selection_method": ["maximum validation F1"],
}).to_csv(output_dir / "day4_threshold_selection.csv", index=False)
print(f"\nSaved holdout metrics, capacity analysis, and threshold selection to {output_dir}.")

,Model,Accuracy,Precision,Recall,F1,TN,FP,FN,TP,PR_AUC,ROC_AUC,Brier_score
0,Majority baseline,0.703,0.000,0.000,0.000,313,0,132,0,NaN,NaN,NaN
1,Logistic Regression (0.50),0.924,0.895,0.841,0.867,300,13,21,111,0.955,0.974,0.051
2,Logistic Regression (0.400),0.926,0.878,0.871,0.875,297,16,17,115,0.955,0.974,0.051



Selected-threshold test confusion matrix [TN, FP; FN, TP]:
[[297  16]
 [ 17 115]]

Test-set outcomes when targeting the highest-risk customers:


,Targeted_share,Customers_targeted,Precision_at_capacity,Churners_captured,Churn_recall_at_capacity
0,0.1,45,1.000,45,0.341
1,0.2,89,0.989,88,0.667
2,0.3,134,0.881,118,0.894



Saved holdout metrics, capacity analysis, and threshold selection to ..\data\processed.


## Refit and save the local scoring model

After holding out the final test set for evaluation, refit the same pipeline on all available labeled customers for offline scoring. The threshold remains the one selected on validation data. The saved model is a local deliverable, not a deployment approval; obtain independent future-period validation before operational use.

In [19]:
import json
import joblib
from pathlib import Path
from sklearn.base import clone

model_output_dir = Path("../outputs")
model_output_dir.mkdir(parents=True, exist_ok=True)

final_model_full = clone(final_model).fit(X_final, y_final)
model_bundle = {
    "model": final_model_full,
    "threshold": selected_threshold,
    "feature_columns": list(X_final.columns),
    "recent_activity_columns": recent_cols,
    "positive_label": 1,
}
model_path = model_output_dir / "churnguard_model.joblib"
joblib.dump(model_bundle, model_path)

metadata = {
    "model_name": "Logistic Regression",
    "training_rows": int(len(X_final)),
    "positive_class": 1,
    "decision_threshold": float(selected_threshold),
    "threshold_selection": "maximum F1 on validation split; provisional",
    "validation_f1_at_selected_threshold": float(best_f1),
    "holdout_test_metrics": final_metrics_df.iloc[2].to_dict(),
    "validation_design": "stratified random 60/20/20 split; not temporal validation",
    "label_observation_end_date": "2024-06-30",
    "features": list(X_final.columns),
    "recent_activity_missing_indicator": "RECENT_ACTIVITY_MISSING",
    "warning": "Validate on a later independently labeled period before operational use.",
}
(model_output_dir / "churnguard_model_metadata.json").write_text(
    json.dumps(metadata, indent=2, allow_nan=False), encoding="utf-8"
)
print(f"Saved full-data scoring model: {model_path}")
print(f"Saved model metadata: {model_output_dir / 'churnguard_model_metadata.json'}")

Saved full-data scoring model: ..\outputs\churnguard_model.joblib
Saved model metadata: ..\outputs\churnguard_model_metadata.json


## One-customer scoring example

For a reproducible demo, randomly select one existing customer profile and hide its churn label before scoring. The feature values come from the supplied data; this is a demonstration, not an independent model evaluation. Run after the model-artifact cell.

In [21]:
demo_customer = model_eligible.sample(n=1, random_state=20261027).copy()
demo_input = demo_customer.drop(
    columns=["CHURNED", "INSUFFICIENT_HISTORY_FLAG"], errors="ignore"
)
demo_input_path = model_output_dir / "single_customer_demo_input.csv"
demo_input.to_csv(demo_input_path, index=False)

demo_features = demo_input[
    [column for column in model_bundle["feature_columns"]
     if column != "RECENT_ACTIVITY_MISSING"]
].copy()
demo_features["RECENT_ACTIVITY_MISSING"] = (
    demo_features[model_bundle["recent_activity_columns"]]
    .isna().all(axis=1).astype(int)
)
demo_features = demo_features[model_bundle["feature_columns"]]
demo_probability = model_bundle["model"].predict_proba(demo_features)[0, 1]
demo_threshold = model_bundle["threshold"]
demo_prediction = int(demo_probability >= demo_threshold)
demo_result = pd.DataFrame([{
    "CUSTOMER_ID": demo_input["CUSTOMER_ID"].iloc[0],
    "CHURN_RISK_SCORE": demo_probability,
    "PREDICTED_CHURN": demo_prediction,
    "DECISION_THRESHOLD": demo_threshold,
    "RECOMMENDED_ACTION": (
        "Send a personalized coupon" if demo_prediction
        else "Continue regular engagement"
    ),
}])
demo_result.to_csv(
    model_output_dir / "single_customer_demo_output.csv", index=False
)

print("One randomly selected example; churn label hidden from scoring.")
display(demo_input)
print("Model output:")
display(demo_result.round(4))
print(f"Saved one-row input and output under {model_output_dir}.")

One randomly selected example; churn label hidden from scoring.


,CUSTOMER_ID,SIGNUP_DATE,AGE,GENDER,CITY,MEMBERSHIP_TIER,MARKETING_OPT_IN,PREFERRED_STORE_ID,PREFERRED_CATEGORY,HOME_STORE_PRICE_TIER,...,AVG_BASKET_VALUE_AVG_PRE,PROMO_TXN_PCT_AVG_PRE,APP_SESSIONS_AVG_PRE,EMAILS_OPENED_AVG_PRE,SUPPORT_TICKETS_AVG_PRE,ACTIVE_MONTHS_PRE,RECENT_TRANSACTIONS,RECENT_SPEND,RECENT_APP_SESSIONS,RECENT_SUPPORT_TICKETS
1828,502135,2023-07-08,52.0,M,Denver,Silver,1,115,Frozen Foods,Unknown,...,31.3775,0.2725,2.375,1.375,0.0,8.0,12.0,381.54,8.0,0.0


Model output:


,CUSTOMER_ID,CHURN_RISK_SCORE,PREDICTED_CHURN,DECISION_THRESHOLD,RECOMMENDED_ACTION
0,502135,0.0004,0,0.3996,Continue regular engagement


Saved one-row input and output under ..\outputs.
